In [1]:
import sys
sys.path.append('../src') # include the src directory


In [2]:

from transformers import AutoTokenizer
import pandas as pd
import data_splitter
import dataset_builder

/Users/sonor/Documents/Projekty/deep-skim/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
2025-07-09 09:35:34,603 - INFO - PyTorch version 2.7.1 available.


In [3]:
file_path = '../data/processed/amyloid-02-07-2025.csv'
df = pd.read_csv(file_path)
print(df.shape)
df = df.dropna(subset=['Abstract']).reset_index(drop=True)  
print(df.shape)



(1423, 9)
(1408, 9)


In [4]:
target = 'rejection'
splitter = data_splitter.DataSplitter(data_splitter.TrainTestSplit(test_size=0.2))
folds = splitter.split(df, target)

2025-07-09 09:35:34,826 - INFO - Splitting data using the selected strategy.
2025-07-09 09:35:34,827 - INFO - Performing stratified train-test split.
2025-07-09 09:35:34,830 - INFO - Train-test split completed.


In [ ]:
model_name = "cambridgeltl/SapBERT-from-PubMedBERT-fulltext"
tokenizer = AutoTokenizer.from_pretrained(model_name)

In [ ]:

SPECIAL_TOKEN = "[KEY]"

tokenizer.add_special_tokens({
"additional_special_tokens": [SPECIAL_TOKEN]
})

3

In [ ]:
keywords = ["aggregates", "amyloid"]
tts_converter = dataset_builder.TrainTestConverter()
hf_dataset = tts_converter.convert(folds, tokenizer, keywords)


Map: 100%|██████████| 282/282 [00:00<00:00, 6235.50 examples/s]
2025-07-09 09:35:35,774 - INFO - Converted train-test split into a single DatasetDict with 'train' and 'test' Datasets.


In [11]:
hf_dataset

DatasetDict({
    train: Dataset({
        features: ['PMID', 'reason', 'decision', 'Title', 'Abstract', 'Authors', 'Journal', 'References', 'labels', '__index_level_0__', 'input_ids', 'token_type_ids', 'attention_mask', 'Abstract2'],
        num_rows: 1126
    })
    test: Dataset({
        features: ['PMID', 'reason', 'decision', 'Title', 'Abstract', 'Authors', 'Journal', 'References', 'labels', '__index_level_0__', 'input_ids', 'token_type_ids', 'attention_mask', 'Abstract2'],
        num_rows: 282
    })
})

In [8]:
splitter.set_strategy(data_splitter.StratifiedKFoldSplit(n_splits=5, shuffle=True, random_state=42))
folds = splitter.split(df, target)

2025-07-09 09:35:35,778 - INFO - Switching data splitting strategy.
2025-07-09 09:35:35,779 - INFO - Splitting data using the selected strategy.
2025-07-09 09:35:35,779 - INFO - Performing Stratified K-Fold split (5 folds)
2025-07-09 09:35:35,783 - INFO - Stratified K-Fold split completed.


In [9]:
cv_converter = dataset_builder.FoldsConverter(tts_converter)
hf_cv_dataset = cv_converter.convert(folds, tokenizer, keywords)

Map: 100%|██████████| 282/282 [00:00<00:00, 6084.67 examples/s]
2025-07-09 09:35:36,024 - INFO - Converted train-test split into a single DatasetDict with 'train' and 'test' Datasets.
2025-07-09 09:35:36,024 - INFO - Fold 0: conversion finished.
Map: 100%|██████████| 282/282 [00:00<00:00, 6313.08 examples/s]
2025-07-09 09:35:36,257 - INFO - Converted train-test split into a single DatasetDict with 'train' and 'test' Datasets.
2025-07-09 09:35:36,257 - INFO - Fold 1: conversion finished.
Map: 100%|██████████| 282/282 [00:00<00:00, 6388.75 examples/s]
2025-07-09 09:35:36,493 - INFO - Converted train-test split into a single DatasetDict with 'train' and 'test' Datasets.
2025-07-09 09:35:36,493 - INFO - Fold 2: conversion finished.
Map: 100%|██████████| 281/281 [00:00<00:00, 6272.85 examples/s]
2025-07-09 09:35:36,729 - INFO - Converted train-test split into a single DatasetDict with 'train' and 'test' Datasets.
2025-07-09 09:35:36,729 - INFO - Fold 3: conversion finished.
Map: 100%|██████

In [10]:
hf_cv_dataset

[DatasetDict({
     train: Dataset({
         features: ['PMID', 'reason', 'decision', 'Title', 'Abstract', 'Authors', 'Journal', 'References', 'labels', '__index_level_0__', 'input_ids', 'token_type_ids', 'attention_mask', 'Abstract2'],
         num_rows: 1126
     })
     test: Dataset({
         features: ['PMID', 'reason', 'decision', 'Title', 'Abstract', 'Authors', 'Journal', 'References', 'labels', '__index_level_0__', 'input_ids', 'token_type_ids', 'attention_mask', 'Abstract2'],
         num_rows: 282
     })
 }),
 DatasetDict({
     train: Dataset({
         features: ['PMID', 'reason', 'decision', 'Title', 'Abstract', 'Authors', 'Journal', 'References', 'labels', '__index_level_0__', 'input_ids', 'token_type_ids', 'attention_mask', 'Abstract2'],
         num_rows: 1126
     })
     test: Dataset({
         features: ['PMID', 'reason', 'decision', 'Title', 'Abstract', 'Authors', 'Journal', 'References', 'labels', '__index_level_0__', 'input_ids', 'token_type_ids', 'attention_m